# 01. The data: Ken French's library, downloaded and counted

**Terms used in this notebook.** Vintage: the version of Ken French's files on the download date, named by the release of the CRSP database they were built from (202607 is the July 2026 release). Provenance: the record of what was downloaded, when, with its checksum and vintage. Factor: a return series that moves many assets at once, for example the return of the whole market; the Fama-French factors SMB and HML are the returns of small firms over large ones and of cheap firms over expensive ones. Risk-free rate: the return on a one-month US Treasury bill, the closest thing to a return with no risk. Excess return: a return minus the risk-free rate over the same month. Estimation window: the M months of past returns a rule sees when it forms its weights, 120 here. Rolling evaluation: moving the window forward one month at a time, forming the weights, and recording the return of the month after the window. Out-of-sample: measured on months the rule had not seen when it formed its weights. In-sample: measured on the same months that were used to form the weights. Sharpe ratio: the average monthly return above the risk-free rate, divided by the standard deviation of that return; the reward earned per unit of risk taken. Turnover: the fraction of the portfolio bought and sold at a rebalance, summed over the assets. Band: the interval, fixed before any code ran, within which a replicated number counts as matching the published one, 0.03 for a Sharpe ratio and 25% for turnover. Gated: said of a cell the band applies to. Verdict: pass or MISS for a gated cell.

**What this notebook does.** It downloads the six Ken French files this project uses, records what it downloaded (date, URL, SHA-256 of the zip, and the CRSP vintage French prints at the top of each file), parses each file into monthly tables, and counts what is in them against what DeMiguel, Garlappi and Uppal (2009) say they used.

**Why it comes first.** Every number in the project is computed from these files, and French revises history: a return from 1985 can differ between the file served today and the one served last year. A replication that misses a published number by 0.01 has to be able to say whether the code or the vintage moved. The provenance record written here makes that possible, and it is attached to every later output.

**What I expect to see, and where it comes from.**

- The three portfolio files (10 industries, 49 industries, 25 size-and-value portfolios) and the three-factor file run monthly from 1926-07. On the vintage I read on 10 September 2026 (CRSP 202607) that is 1,201 rows to 2026-07; a later vintage adds one row per month. The five-factor file starts in 1963-07 (757 rows on that vintage) and the momentum factor in 1927-01.
- DGU's sample is 1963-07 to 2004-11 (their Table 2), which is **497 months**. With their 120-month estimation window the out-of-sample period is 1973-07 to 2004-11, **377 months**. Both counts are asserted below.
- Their four French-sourced datasets have N = 11, 3, 21 and 24 assets (Table 2). The 21 and 24 come from 20 of the 25 size-and-value portfolios: DGU drop the five largest-size portfolios (their footnote 24), because the market, SMB and HML are almost a linear combination of all 25.
- In French's 49-industry file nine industries have gaps coded -99.99 in the early decades. All 49 are populated from **1969-07**; the extension in later notebooks starts there.
- One number that is already a replication check: DGU's Table 3 reports the Sharpe ratio of the value-weighted market, "vw", as 0.1138 in every French-sourced column. That is the monthly Sharpe ratio of French's Mkt-RF over the out-of-sample window. The same calculation on today's vintage is printed beside it. The two differ because the vintages differ, and the size of the difference shows how much revision the sample has absorbed since 2009.

## The constants file

Every band, gate, sample boundary and design parameter the project uses lives in one module, `src/bp/constants.py`, with the source of each number beside it. The notebooks read these names and never introduce numbers of their own. The cell below writes that module into the runtime so this notebook runs from scratch; in the repository the file is the same text.

In [ ]:
import os
os.makedirs("src/bp", exist_ok=True)
open("src/bp/__init__.py", "w").close()

In [ ]:
%%writefile src/bp/constants.py
"""
Pre-committed numbers for the study.

Every band, gate, sample boundary and design parameter is defined here,
one per line, with its source beside it. A notebook may read these names; it
may not introduce numbers of its own. The replication bands (section 3)
and the design of the extension (section 5) were fixed on 10 September 2026,
before any of the rules' code was written. Three entries were changed after that
date; each carries the date of the change, the evidence for it, and what it
replaced, so that a reader can judge whether the change was a specification
settled by the paper's own text or a number tuned to the results.

DGU throughout means DeMiguel, Garlappi and Uppal (2009), "Optimal Versus
Naive Diversification: How Inefficient is the 1/N Portfolio Strategy?",
Review of Financial Studies 22(5), 1915-1953. Page numbers are the journal's.
"""

# ---------------------------------------------------------------------------
# 1. Data: Ken French's library, downloaded at run time, never committed
# ---------------------------------------------------------------------------

FRENCH_BASE_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"

# Every file the study reads. The vintage is whatever the library serves on the
# download date; the loader records the date, the SHA-256 of the zip and the
# CRSP vintage line printed at the top of each CSV.
FRENCH_FILES = {
    "factors3":   "F-F_Research_Data_Factors_CSV.zip",        # Mkt-RF, SMB, HML, RF, monthly from 1926-07
    "momentum":   "F-F_Momentum_Factor_CSV.zip",              # Mom, monthly from 1927-01
    "factors5":   "F-F_Research_Data_5_Factors_2x3_CSV.zip",  # Mkt-RF, SMB, HML, RMW, CMA, RF from 1963-07
    "ind10":      "10_Industry_Portfolios_CSV.zip",           # DGU "Industry" dataset
    "ind49":      "49_Industry_Portfolios_CSV.zip",           # the extension universe; carries firm counts and average size
    "size_bm25":  "25_Portfolios_5x5_CSV.zip",                # DGU "FF-1-factor" and "FF-4-factor" datasets
}

FRENCH_MISSING_CODES = (-99.99, -999.0)   # French's own convention, stated at the top of every portfolio file
FRENCH_RETURNS_ARE_PERCENT = True         # the loader divides by 100 once, at load time, and nowhere else

# ---------------------------------------------------------------------------
# 2. The DGU replication
# ---------------------------------------------------------------------------

DGU_SAMPLE_START = "1963-07"   # DGU Table 2, p. 1918, for every French-sourced dataset
DGU_SAMPLE_END = "2004-11"     # DGU Table 2, p. 1918
DGU_ESTIMATION_WINDOW = 120    # months; DGU section 2, p. 1927. M = 60 is in DGU's appendix and is out of scope.
DGU_RISK_AVERSION = 1.0        # gamma in the CEQ, DGU p. 1929: "the results we report are for the case of gamma = 1"
DGU_TCOST = 0.0050             # proportional cost per unit of one-way turnover; DGU p. 1929: 50 basis points, after Balduzzi and Lynch (1999)
TCOST_SENSITIVITY = 0.0100     # the one cost sensitivity reported: double the base cost
DGU_GMINC_LOWER_BOUND_FRACTION = 0.5   # g-min-c imposes w >= a * 1 with a = 1/(2N); DGU p. 1926
DGU_BS_COV_DOF = "M - N - 2"   # the Bayes-Stein covariance divides by M - N - 2; DGU equation (5), p. 1923

# Changed 22 September 2026: a specification the paper leaves ambiguous, settled
# by its own footnote and by replication. DGU's Lagrangian (8), p. 1925, shows
# the short-sale constraint alone, which with the normalisation of equation (1)
# would make gamma irrelevant. Footnote 22, p. 1934, says the constrained rules
# produce "corner solutions with all wealth invested in a single asset", which
# only happens when the budget constraint 1'w = 1 sits inside the optimisation
# with gamma = 1. Both readings were run: with the budget inside, mv-c on
# MKT/SMB/HML gives a Sharpe ratio of 0.1093 against the published 0.1084 and
# puts all wealth in one asset in 257 of 377 months; without it, 0.2382 and
# never. The other three datasets and the turnover and CEQ rows agree the same
# way. The reading that reproduces the paper's tables is used; the literal
# reading is kept as rules.mv_c_cone so the comparison can be rerun
# (notebook 03), and is not a reported rule.
DGU_CONSTRAINED_BUDGET_INSIDE = True

# The rules replicated: those whose only inputs are the window's
# sample moments (and, for Bayes-Stein, Jorion's prior built from them), plus
# the value-weighted market. g-min-c is min-c with the lower bound raised from
# 0 to 1/(2N) (DGU p. 1926). vw is included because its Sharpe ratio is the
# same 0.1138 in every French-sourced column of Table 3, which makes it a free
# check that the market series and the sample match the paper's.
DGU_RULES = ("ew", "mv", "bs", "min", "vw", "mv-c", "bs-c", "min-c", "g-min-c")
DGU_IN_SAMPLE_ROW = "mv (in sample)"   # DGU Table 3 row 2: mean-variance with M = T, the upper bound estimation error costs

# Excluded, each with the criterion. They are not built.
DGU_RULES_EXCLUDED = {
    "Bayesian diffuse-prior": "not reported by DGU (Table 1 note, p. 1917)",
    "dm": "Bayesian Data-and-Model (Pastor 2000); needs an asset-pricing prior, outside the moment-based set",
    "mp": "MacKinlay and Pastor (2000) missing-factor model; needs a maximum-likelihood factor structure, outside the moment-based set",
    "mv-min": "Kan and Zhou (2007) three-fund mixture of two rules already built; not part of the pre-specified set",
    "ew-min": "mixture of 1/N and minimum variance; not part of the pre-specified set",
    "multi-prior": "not reported by DGU (Table 1 note, p. 1917)",
}

# The datasets replicated: the four DGU built from French's site (DGU Table 2, p. 1918; Appendix A, pp. 1948-1949).
# Each is monthly excess returns over the T-bill rate from French's factor file.
DGU_DATASETS = {
    "Industry":     {"n_risky": 10, "factors": ("Mkt-RF",), "N": 11, "source": "ind10 + factors3"},
    "MKT/SMB/HML":  {"n_risky": 0,  "factors": ("Mkt-RF", "SMB", "HML"), "N": 3, "source": "factors3"},
    "FF-1-factor":  {"n_risky": 20, "factors": ("Mkt-RF",), "N": 21, "source": "size_bm25 + factors3"},
    "FF-4-factor":  {"n_risky": 20, "factors": ("Mkt-RF", "SMB", "HML", "Mom"), "N": 24, "source": "size_bm25 + factors3 + momentum"},
}
DGU_DATASETS_EXCLUDED = {
    "S&P Sectors": "Roberto Wessels' GICS sector data, not on French's site (DGU Appendix A.1)",
    "International": "MSCI country indices, not on French's site (DGU Appendix A.3)",
    "FF-3-factor": "DGU do not report it: 'very similar to FF-1-factor' (Table 3 note, p. 1931)",
}
DGU_EXCLUDE_LARGEST_SIZE_QUINTILE = True   # DGU footnote 24, p. 1949: the five largest-size portfolios of the 25 are dropped, leaving 20

# ---------------------------------------------------------------------------
# 3. Bands for the replication, written before the code ran
# ---------------------------------------------------------------------------

BAND_SHARPE_ABS = 0.03          # out-of-sample Sharpe ratio within 0.03 of DGU Table 3, per rule and dataset
CHECK_TOP3_RANKING = True        # the ranking of the top three rules within each dataset is preserved
BAND_TURNOVER_REL = 0.25        # turnover within 25% relative of DGU Table 5

# DGU Table 5 reports the unconstrained mean-variance rule's turnover at 606,594 times
# that of 1/N on the Industry dataset and 10,466 times on FF-1-factor. Those
# figures are dominated by the months in which the sample covariance is
# near-singular, and they move by orders of magnitude when French revises
# a single return. A 25% band on them tests the vintage and says nothing about
# the code. The band
# is therefore applied only where DGU's published relative turnover is below
# the value here; above it, the replicated figure is reported beside the
# published one with no pass or fail. Fixed 10 September 2026, before any
# of the rules' code existed.
BAND_TURNOVER_APPLIES_BELOW_RELATIVE = 100.0

# Changed 13 September 2026, after notebook 02 had run, on its evidence. The
# out-of-sample Sharpe band applies only to rules whose published relative
# turnover (DGU Table 5, panel A) is below the same level; a rule above it is
# reported beside the published figure with the range that ten basis points of
# noise produces, and carries no verdict. The level is the paper's own number
# and the turnover band already used it, so no new level enters here. The
# in-sample mean-variance row carries no verdict either: BAND_SHARPE_ABS was
# written for out-of-sample Sharpe ratios. Evidence: on the Industry, FF-1 and
# FF-4 datasets the unconstrained mean-variance rule moved by 0.12 to 0.25
# under ten basis points of noise, with the published figure inside the range,
# while the 1/N rows matched to the third decimal. Before the change the band
# applied to every out-of-sample row.
BAND_SHARPE_APPLIES_BELOW_RELATIVE = BAND_TURNOVER_APPLIES_BELOW_RELATIVE
BAND_SHARPE_DECISION_DATE = "2026-09-13"

# Sharpe ratios are compared on the same sample, 1963-07 to 2004-11, 120-month
# window, so the out-of-sample period is 1973-07 to 2004-11.
DGU_EXPECTED_T = 497           # months from 1963-07 to 2004-11 inclusive; a count, checked in notebook 01
DGU_EXPECTED_OOS = 377         # DGU_EXPECTED_T - DGU_ESTIMATION_WINDOW

# The replication extended to the latest month (notebook 07). Fixed 1 October
# 2026, before the notebook was written. The nine rules run through the same
# rolling evaluation on every month the vintage carries, from DGU_SAMPLE_START
# to its last month, and the measures are reported for three periods: the
# paper's out-of-sample period (1973-07 to 2004-11, the numbers of notebooks
# 02 and 03), the months since the paper (EXTENDED_NEW_PERIOD_START to the
# last month, every one of them out of sample for a window that begins 120
# months earlier), and the whole period. No published number exists for the
# new months, so the pre-committed statements are expectations, each reported
# as a count.
EXTENDED_NEW_PERIOD_START = "2004-12"   # the month after DGU's sample ends
# Expectations:
#  1. Count first: every dataset carries a return for every month from
#     DGU_SAMPLE_START to the vintage's last month, the four datasets end in
#     the same month, and the number of months is printed and recorded.
#  2. The paper's reading of Table 3 holds on the whole period: each
#     short-sale-constrained rule has a higher out-of-sample Sharpe ratio than
#     its unconstrained version on every dataset (mv-c against mv, bs-c
#     against bs, min-c against min; 12 comparisons).
#  3. Sample-based mean-variance is below 1/N on every dataset in the new
#     period and in the whole period (8 cells). The new months add about 260
#     to a problem that the formula of notebook 04 says needs thousands, so
#     nothing else is expected to change.
#  4. The number of cells in the new period in which a rule beats 1/N with a
#     Jobson-Korkie p-value below 0.05 is reported. On about 260 months the
#     standard error of a Sharpe-ratio difference is about 0.06, so few or
#     none are expected, and none would be claimed as a finding.
# Turnover inside a sub-period counts every rebalance that falls in it; the
# first out-of-sample month of the whole period has no rebalance, as in DGU.

# ---------------------------------------------------------------------------
# 4. DGU's analytical result and simulation (notebook 04)
# ---------------------------------------------------------------------------

# The analytical critical window (DGU Proposition 1, pp. 1937-1938; Figure 1, p. 1940).
# Panel E is the one calibrated to US stock-market data and the one the abstract quotes.
ANALYTIC_SHARPE_TANGENCY = 0.15    # S*, DGU Figure 1 panel E
ANALYTIC_SHARPE_1N = 0.12          # S_ew, DGU Figure 1 panel E
EXPECT_CRITICAL_M_LOWER = {25: 3000, 50: 6000}   # DGU p. 1941: "more than 3000 months" and "more than 6000 months"; the check is order of magnitude and direction in N

# The simulated dataset (DGU section 5.1, pp. 1941-1942).
SIM_RF_MEAN_ANNUAL = 0.02
SIM_RF_SD_ANNUAL = 0.02
SIM_FACTOR_MEAN_ANNUAL = 0.08
SIM_FACTOR_SD_ANNUAL = 0.16
SIM_ALPHA = 0.0
SIM_BETA_RANGE = (0.5, 1.5)          # loadings evenly spread
SIM_IDIO_VOL_RANGE = (0.10, 0.30)    # annual idiosyncratic volatility, uniform, cross-sectional mean 0.20
SIM_N = (10, 25, 50)
SIM_M = (120, 360, 6000)
SIM_T = 24000                        # months in one simulated history
SIM_SEED = 20260910                  # fixed so the run is reproducible; DGU's seed is unknown
# In one 24,000-month simulated history the standard error of a monthly Sharpe ratio is about
# 1/sqrt(24000) = 0.0065, so the comparison with DGU Table 6 uses the same band
# as the empirical tables, and in addition the sign of (mv minus 1/N) is checked
# against Table 6 in every (N, M) cell.
BAND_SIM_SHARPE_ABS = BAND_SHARPE_ABS
# Added 29 September 2026, when notebook 04 was written. Table 6 reports 1/N and
# "mv (true)" as constants across M within each N, so they are full-sample
# Sharpe ratios; the estimated rules are out of sample. The comparison follows
# that convention. As a second check, every rule is rerun on SIM_SPREAD_DRAWS
# alternative draws of the market (seeds 1 to SIM_SPREAD_DRAWS) and the
# published value is checked against the range they produce, the simulation's
# analogue of the fragility test in notebook 02.
SIM_SPREAD_DRAWS = 8

# Fat tails (notebook 05), an addition to the paper. Fixed 30 September 2026,
# before the notebook was written. The market above is rerun with every shock
# drawn from a Student-t distribution instead of a normal one, scaled so that
# the mean and covariance of returns are exactly those of the normal market:
# only the tails differ, and the tangency portfolio, S*, S_ew and the formula's
# critical window are the same numbers as in notebook 04. One scale draw per
# month is shared by the factor shock and every idiosyncratic shock (a
# multivariate t), so an extreme month hits all assets at once, which is how fat
# tails appear in markets. Months stay independent of one another; the
# normality assumption is relaxed and nothing else. The normal shocks are the
# same draws as notebook 04's (same seed), so each fat-tailed market is paired
# with its normal twin month by month.
SIM_T_DOF = (10, 5)                  # degrees of freedom. 10: excess kurtosis 1, about a broad index's monthly returns. 5: excess kurtosis 6, about an industry portfolio's. At 4 or below the fourth moment is infinite, the standard error of a Sharpe ratio is undefined, and no comparison could be read.
SIM_T_COMMON_SCALE = True            # one scale per month shared by all shocks (multivariate t); False would draw one scale per asset
# The crossing table: sample-based mean-variance against 1/N over a grid of
# windows, both measured on the same out-of-sample months, to locate the
# window at which the gap closes under each distribution. The formula's
# critical window for the market's own S* and S_ew is reported beside it.
SIM_CROSSING_WINDOWS = (120, 240, 360, 600, 1000, 2000, 3000, 4000, 6000, 9000, 12000)
# Expectations written before the run, each reported as a count of cells.
#  1. 1/N and mv (true), which use no estimates, have the same full-sample
#     Sharpe ratio under t shocks as under normal ones to within two standard
#     errors; a failure here would be a construction error.
#  2. Every estimated rule's Sharpe ratio in every (N, M) cell is no higher
#     under t shocks than under normal ones, beyond a slack of about one and a
#     half standard errors; the loss is expected to grow as the tails fatten
#     (5 against 10) and as the window shortens.
#  3. For N = 10 the crossing window is later under t shocks than under normal
#     ones. For N = 25 and 50, 1/N is so close to the tangency portfolio in
#     this market that no crossing is expected within 12,000 months under any
#     distribution, and the formula says so.
FAT_TAIL_MOMENT_CHECK_ABS = 0.013    # expectation 1: two standard errors of a 24,000-month Sharpe ratio
FAT_TAIL_DIRECTION_SLACK = 0.01      # expectation 2: a cell counts as in the expected direction if Sharpe(t) <= Sharpe(normal) + this

# Volatility that changes through time (notebook 06), the second addition to
# the paper. Fixed 1 October 2026, before the notebook was written. The market
# of notebook 04 is rerun with shocks whose volatility follows a GARCH(1,1)
# process (Bollerslev 1986), the standard model of volatility clustering: a
# large shock this month raises the volatility of next month's shock, and the
# effect decays slowly. The factor shock and every asset's idiosyncratic shock
# get their own, independent volatility process with the same parameters, so
# both the level and the shape of the covariance matrix move through time:
# when the factor's volatility is high relative to the assets' own, the
# correlations between assets are high, which is what markets do in a crisis.
# The unconditional mean and covariance are exactly those of the normal market
# (omega = 1 - alpha - beta puts the unconditional variance of each scaled
# shock at one), so the tangency portfolio, S*, S_ew and the formula's
# critical window are unchanged. The normal shocks are the same draws as
# notebook 04's, so the two markets are paired month by month. Months are no
# longer independent; the tails stay those of a normal distribution except for
# what the clustering itself produces (excess kurtosis 6 alpha^2 / (1 - beta^2
# - 2 alpha beta - 3 alpha^2), about 0.8 for these parameters).
SIM_GARCH_ALPHA = 0.10               # weight of last month's squared shock; a textbook monthly value
SIM_GARCH_BETA = 0.85                # weight of last month's variance; alpha + beta = 0.95 is the persistence, a half-life of a volatility shock of about 13 months
SIM_GARCH_OWN_PROCESS_PER_ASSET = True   # the factor and each idiosyncratic shock follow their own process; False would scale all shocks by one common volatility, which leaves the shape of the covariance unchanged and the rules' weights with it
# Expectations written before the run, each reported as a count of cells.
#  1. 1/N and mv (true) keep their full-sample Sharpe ratios to within
#     FAT_TAIL_MOMENT_CHECK_ABS of the normal market's; a failure is a
#     construction error.
#  2. Every estimated rule's Sharpe ratio in every (N, M) cell is no higher
#     under volatility clustering than in the normal market, beyond
#     FAT_TAIL_DIRECTION_SLACK. Unlike notebook 05, the loss is expected to be
#     visible, and largest for the rules that depend on the shape of the
#     covariance (min, min-c, g-min-c), because a 120-month average of a
#     covariance that moves is wrong in a known direction for the month that
#     follows.
#  3. For N = 10 the crossing window is later than in the normal market; for
#     N = 25 and 50 no crossing within 12,000 months under either market.
# Descriptive checks reported beside them: the first-order autocorrelation of
# the factor's squared return is about alpha (1 - alpha beta - beta^2) /
# (1 - 2 alpha beta - beta^2) = 0.18 under clustering and zero in the normal
# market, and the average correlation between assets is higher in the months
# of highest factor volatility than in the months of lowest.

# ---------------------------------------------------------------------------
# 5. The extension: constrained index tracking on the 49 industries
# ---------------------------------------------------------------------------

EXT_UNIVERSE = "ind49"
EXT_ESTIMATION_WINDOW = 120        # the same window as the replication, so one convention runs through the repository
EXT_REBALANCE = "monthly"
# The extension sample starts at the first month in which all 49 industries carry
# a return (French codes the early gaps as -99.99). That start month is a count:
# on the 202607 vintage, read on 10 September 2026, Hlth is the last industry to
# fill and does so in 1969-07. Notebook 01 asserts it on every run, so a vintage
# that moves it fails loudly.
EXT_SAMPLE_START = "1969-07"
EXT_SAMPLE_END = None              # the latest month the vintage carries

# The benchmark (the index the portfolio tracks). Count-first check: the
# cap-weighted combination of the 49 industries, weights from firm count times
# average firm size, must track the market factor plus the risk-free rate within
# 50 basis points a year of tracking error over the full sample. If it does not,
# the derived weights are dropped and the benchmark is the market series.
CAPW_TE_MAX_ANNUAL = 0.0050

# Constraints. Fixed 10 September 2026.
ACTIVE_WEIGHT_BOUND = 0.02         # each untilted industry within 2 percentage points of its benchmark weight; an absolute cap does not fit 49 industries whose benchmark weights run from under 0.5% to over 10%
TURNOVER_CAP_MONTHLY_ONE_WAY = 0.02   # one-way turnover per rebalance, against drifted weights; 2% a month is about 24% a year, the enhanced-indexing range
TILT_INDUSTRIES = ("Coal", "Oil", "Util")   # French's column names, stripped of padding
TILT_MAX_SHARE_OF_BENCHMARK = 0.5  # each tilted industry held at no more than half its benchmark weight, reported as a 50% reduction relative to the benchmark; a tilted industry's bounds are [0, 0.5 * benchmark], and ACTIVE_WEIGHT_BOUND does not apply to it, because the two would conflict for any industry above 4% of the benchmark
LONG_ONLY = True

# Constraint sets, cumulative, reported in this order.
CONSTRAINT_SETS = {
    "C0": ("long_only",),
    "C1": ("long_only", "active_weight_bound"),
    "C2": ("long_only", "active_weight_bound", "turnover_cap"),
    "C3": ("long_only", "active_weight_bound", "turnover_cap", "tilt"),
}
# The optimiser also accepts an exclusion list (weight zero on named assets). It
# is built and tested for later use and is not one of the reported constraint
# sets.

# Covariance estimators. LW_TARGET is provisional until the shrinkage target
# used by the constrained-portfolio literature has been checked before notebook
# 10; if a different Ledoit-Wolf target is standard for constrained portfolios,
# the target follows it and this line records the change.
COV_ESTIMATORS = ("sample", "ledoit_wolf", "factor", "pca")
LW_TARGET = "constant_correlation"     # Ledoit and Wolf (2004a), "Honey, I Shrunk the Sample Covariance Matrix"
FACTOR_MODEL_FACTORS = ("Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom")   # the Fama-French five factors plus momentum; the estimator is B F B' plus a diagonal of residual variances, B estimated in the same estimation window
PCA_N_COMPONENTS = 5                   # five components plus a diagonal idiosyncratic term
PCA_PC1_MARKET_CORR_MIN = 0.95         # the first component's correlation with the cap-weighted market return
PCA_USE_COVARIANCE = True              # covariance, not correlation, of monthly excess returns
PCA_SIGN_RULE = "positive market loading"
PCA_WINDOWS = {                        # three pre-specified windows; loading stability is the finding and carries no pass or fail
    "to_2019":   (None, "2019-12"),
    "2020_2022": ("2020-01", "2022-12"),
    "from_2023": ("2023-01", None),
}

# Attribution: factor-based attribution of each constrained portfolio's
# realised active return against the factors above, contribution by factor
# plus residual, with the identity as the check.
ATTRIBUTION_FACTORS = FACTOR_MODEL_FACTORS
ATTRIBUTION_IDENTITY_MAX_ABS_ERROR = 1e-9        # contributions plus residual minus active return, per month, in return units

# Specification count, written down before the code runs and reported.
# Replication: one specification per rule and dataset. Extension:
EXT_SPEC_COUNT = len(COV_ESTIMATORS) * len(CONSTRAINT_SETS)   # 16 optimised specifications
EXT_COST_LEVELS = (DGU_TCOST, TCOST_SENSITIVITY)              # each reported net of cost at both levels; the cost is applied after the fact and does not enter the optimisation

# ---------------------------------------------------------------------------
# 6. Output conventions
# ---------------------------------------------------------------------------

OUTPUT_DIR = "outputs"
PROVENANCE_FILE = "outputs/provenance_french.json"   # download date, URL, SHA-256 and CRSP vintage line per file
TIMEZONE = "Europe/Amsterdam"

## The loader

A French CSV holds between two and ten blocks, each announced by a title line, then a header row that starts with a comma, then rows that start with a date (six digits for months, four for years). Returns are in percent and missing values are coded -99.99 or -999. The loader splits the file into blocks, indexes each by a monthly `Period`, converts percent to decimal once, replaces the missing codes with NaN, and records provenance for the zip it read.

In [ ]:
%%writefile src/bp/french_loader.py
"""
Ken French data library: download, record, parse.

The library serves each dataset as a zip holding one CSV. The CSV is not a
single table. It is a text file with between two and ten blocks, each announced by a title
line ("  Average Value Weighted Returns -- Monthly"), followed by a header row
that starts with a comma, followed by rows that start with a date: six digits
for monthly data (192607), four for annual (1927). Missing values are coded
-99.99 or -999. Returns are in percent.

This module turns that into pandas DataFrames indexed by a monthly Period, in
decimal returns, with the missing codes replaced by NaN. It also writes a
provenance record for every file it downloads: the URL, the download time, the
SHA-256 of the zip, and the CRSP vintage line French prints at the top of each
CSV. A result is only as reproducible as the vintage it was computed on, and
French revises history, so the record is part of every output.

Nothing here is committed to the repository: the zips live in a local cache
directory that .gitignore excludes.
"""
from __future__ import annotations

import hashlib
import io
import json
import re
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import requests

from . import constants as C

_DATE_ROW = re.compile(r"^\s*(\d{6}|\d{4})\s*,")


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def download(key: str, cache_dir: str | Path = "data/french", force: bool = False) -> dict:
    """Fetch one French zip by its key in constants.FRENCH_FILES.

    Returns the provenance record for the file and writes the zip to cache_dir.
    A cached zip is reused unless force=True, and its record is rebuilt from the
    cached bytes, so the SHA-256 always describes the file actually on disk.
    """
    filename = C.FRENCH_FILES[key]
    url = C.FRENCH_BASE_URL + filename
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / filename
    if force or not path.exists():
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        path.write_bytes(response.content)
        fetched_now = True
    else:
        fetched_now = False
    data = path.read_bytes()
    now_utc = datetime.now(timezone.utc)
    record = {
        "key": key,
        "filename": filename,
        "url": url,
        "fetched_this_run": fetched_now,
        "download_time_utc": now_utc.isoformat(timespec="seconds"),
        "download_time_local": now_utc.astimezone(ZoneInfo(C.TIMEZONE)).isoformat(timespec="seconds"),
        "zip_sha256": sha256_bytes(data),
        "zip_bytes": len(data),
        "vintage_line": _vintage_line(data),
    }
    return record


def _csv_text(zip_bytes: bytes) -> str:
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if len(names) != 1:
            raise ValueError(f"expected one CSV in the zip, found {names}")
        return zf.read(names[0]).decode("latin-1")


def _vintage_line(zip_bytes: bytes) -> str:
    """The first line of the CSV, e.g. 'This file was created using the 202607 CRSP database.'"""
    text = _csv_text(zip_bytes)
    for line in text.splitlines():
        if line.strip():
            first = line.strip()
            return first.split(".")[0] + "." if "." in first else first
    return ""


def parse_blocks(text: str) -> list[dict]:
    """Split a French CSV into its blocks.

    Each block is returned as {"title": str, "frequency": "monthly" | "annual",
    "frame": DataFrame} where the frame is indexed by Period and holds floats in
    the file's own units (percent). The title is the last non-empty line before
    the header row that is not itself a data row.
    """
    lines = text.splitlines()
    blocks = []
    i = 0
    n = len(lines)
    while i < n:
        if _DATE_ROW.match(lines[i]):
            # Walk back to the header row (starts with a comma) and the title above it.
            header_idx = i - 1
            while header_idx >= 0 and not lines[header_idx].lstrip().startswith(","):
                header_idx -= 1
            if header_idx < 0:
                raise ValueError(f"data row at line {i} has no header row above it")
            title_idx = header_idx - 1
            while title_idx >= 0 and not lines[title_idx].strip():
                title_idx -= 1
            title = lines[title_idx].strip() if title_idx >= 0 else ""
            # The factor files and the later blocks of the size-and-value file
            # carry no title line, only prose; a sentence is not a title.
            if not title or title.endswith(".") or len(title) > 70:
                title = f"(untitled block {len(blocks)})"
            columns = [c.strip() for c in lines[header_idx].split(",")[1:]]
            rows = []
            j = i
            while j < n and _DATE_ROW.match(lines[j]):
                rows.append(lines[j])
                j += 1
            raw = pd.read_csv(io.StringIO("\n".join(rows)), header=None, index_col=0)
            raw.columns = columns[: raw.shape[1]]
            date_len = len(str(raw.index[0]).strip())
            if date_len == 6:
                idx = pd.PeriodIndex([pd.Period(f"{str(d)[:4]}-{str(d)[4:6]}", freq="M") for d in raw.index])
                freq = "monthly"
            else:
                idx = pd.PeriodIndex([pd.Period(str(d), freq="Y") for d in raw.index])
                freq = "annual"
            frame = raw.astype(float)
            frame.index = idx
            frame.index.name = "period"
            blocks.append({"title": title, "frequency": freq, "frame": frame})
            i = j
        else:
            i += 1
    return blocks


def load_blocks(key: str, cache_dir: str | Path = "data/french") -> list[dict]:
    """All blocks of one file, with the file's provenance record attached to each."""
    record = download(key, cache_dir)
    text = _csv_text((Path(cache_dir) / C.FRENCH_FILES[key]).read_bytes())
    blocks = parse_blocks(text)
    for b in blocks:
        b["provenance"] = record
    return blocks


def load_monthly(key: str, block: int | str = 0, cache_dir: str | Path = "data/french",
                 as_decimal: bool = True) -> pd.DataFrame:
    """One monthly block of one file, in decimal returns with missing codes as NaN.

    block is an integer position among the file's blocks, or a regular
    expression matched against block titles (the first match wins).
    """
    blocks = load_blocks(key, cache_dir)
    if isinstance(block, int):
        chosen = blocks[block]
    else:
        matches = [b for b in blocks if re.search(block, b["title"], flags=re.I)]
        if not matches:
            raise KeyError(f"no block in {key} matches {block!r}; titles: {[b['title'] for b in blocks]}")
        chosen = matches[0]
    frame = chosen["frame"].copy()
    for code in C.FRENCH_MISSING_CODES:
        frame = frame.mask(np.isclose(frame, code))
    if as_decimal and C.FRENCH_RETURNS_ARE_PERCENT:
        frame = frame / 100.0
    frame.attrs["title"] = chosen["title"]
    frame.attrs["provenance"] = chosen["provenance"]
    return frame


def load_counts(key: str, block: int | str, cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """A block in the file's own units (firm counts, average firm size), missing codes as NaN."""
    return load_monthly(key, block, cache_dir, as_decimal=False)


def write_provenance(records: list[dict], path: str | Path = C.PROVENANCE_FILE) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    ordered = sorted(records, key=lambda r: r["key"])
    path.write_text(json.dumps(ordered, indent=2, sort_keys=True) + "\n")
    return path


def block_summary(key: str, cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """One row per block: title, frequency, columns, rows, first and last period."""
    rows = []
    for k, b in enumerate(load_blocks(key, cache_dir)):
        f = b["frame"]
        rows.append({
            "file": key, "block": k, "title": b["title"], "frequency": b["frequency"],
            "columns": f.shape[1], "rows": f.shape[0],
            "first": str(f.index[0]), "last": str(f.index[-1]),
        })
    return pd.DataFrame(rows)

## Download and record

Six files. The table shows what was fetched and the vintage line French prints at the top of each. The record is written to `outputs/provenance_french.json`, which later notebooks read so that every output can name the vintage it was computed on.

In [ ]:
import sys
sys.path.insert(0, "src")
import pandas as pd
import numpy as np
from bp import constants as C
from bp import french_loader as fl

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 70)

records = [fl.download(key) for key in C.FRENCH_FILES]
path = fl.write_provenance(records)
prov = pd.DataFrame(records)[["key", "filename", "zip_bytes", "zip_sha256", "download_time_local", "vintage_line"]]
prov["zip_sha256"] = prov["zip_sha256"].str[:16] + "..."
print("provenance written to", path)
prov

## What is in each file

One row per block. The first block of each portfolio file is the value-weighted monthly return, which is the one the project uses; the 49-industry file also carries the number of firms and the average firm size per industry per month, which is how the cap weights of the industries are derived in notebook 08.

In [ ]:
summaries = pd.concat([fl.block_summary(key) for key in C.FRENCH_FILES], ignore_index=True)
summaries

## Count first

Each check below is an assertion. If a later vintage moves a count, the cell fails and names the count. A silent change in the sample would be worse than a stopped notebook.

In [ ]:
f3  = fl.load_monthly("factors3", 0)
f5  = fl.load_monthly("factors5", 0)
mom = fl.load_monthly("momentum", 0)
i10 = fl.load_monthly("ind10", 0)
i49 = fl.load_monthly("ind49", 0)
p25 = fl.load_monthly("size_bm25", 0)

# 1. The DGU sample window: 497 months, 377 of them out of sample.
window = slice(pd.Period(C.DGU_SAMPLE_START, "M"), pd.Period(C.DGU_SAMPLE_END, "M"))
T = len(f3.loc[window])
assert T == C.DGU_EXPECTED_T, f"DGU window holds {T} months, expected {C.DGU_EXPECTED_T}"
assert T - C.DGU_ESTIMATION_WINDOW == C.DGU_EXPECTED_OOS
print(f"DGU sample {C.DGU_SAMPLE_START} to {C.DGU_SAMPLE_END}: T = {T} months, out of sample = {T - C.DGU_ESTIMATION_WINDOW}")

# 2. No missing values inside the DGU window, in any file the replication uses.
for name, frame in [("ind10", i10), ("size_bm25", p25), ("factors3", f3), ("momentum", mom)]:
    n_missing = int(frame.loc[window].isna().sum().sum())
    assert n_missing == 0, f"{name} has {n_missing} missing values inside the DGU window"
    assert len(frame.loc[window]) == T, f"{name} has {len(frame.loc[window])} rows in the DGU window"
print("no missing values in the DGU window in ind10, size_bm25, factors3, momentum")

# 3. The 20 of 25: drop the five largest-size portfolios (DGU footnote 24).
big = [c for c in p25.columns if c.startswith("BIG") or c.startswith("ME5")]
assert len(big) == 5, big
p20 = p25.drop(columns=big)
assert p20.shape[1] == 20
print("dropped from the 25:", big)

# 4. Asset counts per DGU dataset, against Table 2.
counts = {
    "Industry":    i10.shape[1] + 1,
    "MKT/SMB/HML": 3,
    "FF-1-factor": p20.shape[1] + 1,
    "FF-4-factor": p20.shape[1] + 4,
}
for name, n in counts.items():
    assert n == C.DGU_DATASETS[name]["N"], (name, n)
print("N per dataset:", counts)

# 5. The 49 industries: which have gaps, and from when all 49 are populated.
gaps = i49.isna().sum()
gaps = gaps[gaps > 0]
first_full = i49.dropna(how="any").index[0]
assert str(first_full) == C.EXT_SAMPLE_START, f"all 49 industries populated from {first_full}, constants say {C.EXT_SAMPLE_START}"
assert not i49.loc[first_full:].isna().any().any()
print("industries with early gaps (months missing):", gaps.to_dict())
print("all 49 industries populated from", first_full, "to", i49.index[-1], "=", len(i49.loc[first_full:]), "months")

# 6. Tilt industries exist under the names the constants use.
assert all(t in i49.columns for t in C.TILT_INDUSTRIES), C.TILT_INDUSTRIES
print("tilt industries present:", C.TILT_INDUSTRIES)

## One replication number, before any rule exists

DGU's "vw" row is the value-weighted market held without trading. Its Sharpe ratio is the mean of monthly Mkt-RF divided by its standard deviation over the out-of-sample months. DGU report 0.1138. The cell prints the same quantity on today's vintage, and for reference on the full 497-month window as well.

In [ ]:
oos = slice(pd.Period(C.DGU_SAMPLE_START, "M") + C.DGU_ESTIMATION_WINDOW, pd.Period(C.DGU_SAMPLE_END, "M"))
mkt_oos = f3.loc[oos, "Mkt-RF"]
mkt_full = f3.loc[window, "Mkt-RF"]
sr_oos = mkt_oos.mean() / mkt_oos.std(ddof=1)
sr_full = mkt_full.mean() / mkt_full.std(ddof=1)
print(f"out-of-sample window {oos.start} to {oos.stop}: {len(mkt_oos)} months")
print(f"Sharpe ratio of Mkt-RF, out of sample : {sr_oos:.4f}   (DGU Table 3, vw: 0.1138)")
print(f"Sharpe ratio of Mkt-RF, full DGU window: {sr_full:.4f}")
print(f"difference to the published 0.1138: {sr_oos - 0.1138:+.4f}  (band for the rules' Sharpe ratios: {C.BAND_SHARPE_ABS})")

## What this notebook established, and what could be wrong

The six files parse, the DGU window holds 497 months with no gaps, the asset counts match Table 2, and all 49 industries are populated from 1969-07. The market Sharpe ratio on today's vintage sits within 0.0025 of the 0.1138 DGU printed, which is the size of revision to expect on the rule rows too.

Three things this does not settle. French's risk-free rate is the one-month Treasury bill; DGU describe theirs as the 90-day bill taken from French's site, and French's site only carries the one-month series, so I use that. The vintage line names the CRSP cut French built the file from and nothing else; two files with the same vintage line can still differ if French changed a method. And the checks above count rows and columns; they do not verify a single return value, which only the replication itself can do.